In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef
import joblib

# 1. Load Data (Path aapke folder structure ke hisaab se)
df = pd.read_csv('../data/data.csv')

# 2. Data Cleaning & Preprocessing (EDA Insights)
# Drop duplicates
df = df.drop_duplicates()

# Clean string columns (remove trailing spaces)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# 3. Drop Leaky and Unnecessary Columns
# 'Unnamed: 13' target ka duplicate hai.
# 'Feedback' leakage create karta hai. 
# 'Pin code', 'latitude', 'longitude' ko is basic model se hata rahe hain (advanced spatial modeling ke bina inhe use karna theek nahi).
cols_to_drop = ['Unnamed: 13', 'Feedback', 'Pin code', 'latitude', 'longitude']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 4. Define Features (X) and Target (y)
X = df.drop(columns=['Output'])
y = df['Output'].apply(lambda x: 1 if x == 'Yes' else 0) # Encode Target: Yes=1, No=0

# 5. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 6. Build Preprocessing Pipeline
# Baaki bache huye categorical columns ko One-Hot Encode karenge
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ], 
    remainder='passthrough' # Age aur Family size (numeric) ko as-is rehne dega
)

# 7. Create Full Model Pipeline (with class_weight='balanced')
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

# 8. Train the Model
model_pipeline.fit(X_train, y_train)

# 9. Evaluate the Model
y_pred = model_pipeline.predict(X_test)

print("\n--- Model Evaluation ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

mcc = matthews_corrcoef(y_test, y_pred)
print(f"Matthews Correlation Coefficient (MCC): {mcc:.3f}")

# 10. Save the Pipeline for future use (train.py me ye logic jayega)
# joblib.dump(model_pipeline, '../models/random_forest_v1.pkl')
# print("\nModel saved successfully to '../models/random_forest_v1.pkl'")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_19720\4263772387.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns
C:\Users\lenovo\AppData\Local\Temp\ipykernel_19720\4263772387.py:44: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/doc

Training data shape: (228, 8)
Testing data shape: (57, 8)

--- Model Evaluation ---
Confusion Matrix:
[[ 7  7]
 [ 8 35]]

Classification Report:
              precision    recall  f1-score   support

      No (0)       0.47      0.50      0.48        14
     Yes (1)       0.83      0.81      0.82        43

    accuracy                           0.74        57
   macro avg       0.65      0.66      0.65        57
weighted avg       0.74      0.74      0.74        57

Matthews Correlation Coefficient (MCC): 0.307
